# Georeferenced Occurrence Records of African Bats Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.1v72-sdaa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Display basic metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"\nPublished: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Keywords: {', '.join(getattr(metadata, 'keywords', []))}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Get all record sets from the metadata
record_sets = list(dataset.list_record_sets())
print("Available Record Sets:")
for record_set in record_sets:
    print(f"- @id: {record_set['@id']} | name: {record_set.get('name', 'N/A')}")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        if isinstance(field, dict):
            field_id = field.get('@id')
            field_name = field.get('name', 'N/A')
        else:
            field_id = field
            field_name = 'N/A'
        print(f"    - @id: {field_id} | name: {field_name}")

In [ ]:
# Preview a few records from each record set via their @id
for record_set in record_sets:
    rec_id = record_set['@id']
    print(f"\nSample from record set @id={rec_id}:")
    for i, record in enumerate(dataset.records(record_set=rec_id)):
        pprint.pprint(record)
        if i > 2:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rec_id in record_set_ids:
    records = list(dataset.records(record_set=rec_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"Loaded DataFrame for record set @id={rec_id} with {len(df)} records and columns: {df.columns.tolist()}")
    else:
        print(f"Record set @id={rec_id} contains no records.")

In [ ]:
# Display first rows from the primary record set
if len(dataframes) > 0:
    # Assume first record set is primary for demo
    primary_rec_id = list(dataframes.keys())[0]
    print(f"Columns in record set @id={primary_rec_id}:")
    print(dataframes[primary_rec_id].columns.tolist())
    dataframes[primary_rec_id].head()
else:
    print("No non-empty dataframes found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA: Filter and normalize latitude/longitude
import numpy as np
import warnings

if len(dataframes) > 0:
    # We continue with the primary record set and guess likely field names
    df = dataframes[primary_rec_id]
    numeric_fields = [col for col in df.columns if col.lower() in ['latitude', 'longitude', 'year'] or np.issubdtype(df[col].dtype, np.number)]
    display_fields = numeric_fields if len(numeric_fields) > 0 else df.columns.tolist()
    print(f"Available numeric fields: {display_fields}")
    # Pick latitude as the numeric field if available, else first numeric
    if 'latitude' in df.columns:
        numeric_field = 'latitude'
    elif len(display_fields) > 0:
        numeric_field = display_fields[0]
    else:
        numeric_field = df.columns[0]

    # Clean and convert if necessary
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    
    # Remove nulls
    valid_df = df[df[numeric_field].notnull()].copy()
    threshold = valid_df[numeric_field].mean() if valid_df[numeric_field].notnull().any() else 0
    filtered_df = valid_df[valid_df[numeric_field] > threshold]

    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display_cols = [numeric_field]
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a reasonable field if available
    possible_groups = [col for col in df.columns if col.lower() in ['country', 'family', 'genus']]
    if len(possible_groups) > 0:
        group_field = possible_groups[0]
        print(f"Grouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field, as_index=False).agg({numeric_field: 'mean'})
        print(grouped_df.head())
    else:
        print("No suitable group field found.")
else:
    print("No suitable dataframe for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
    
    if 'longitude' in df.columns and 'latitude' in df.columns:
        plt.figure(figsize=(6,6))
        plt.scatter(df['longitude'], df['latitude'], alpha=0.6)
        plt.title('Geographic Distribution of Records')
        plt.xlabel('Longitude')
        plt.ylabel('Latitude')
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

**Summary:**
- Successfully loaded and parsed the Georeferenced Occurrence Records of African Bats dataset via the Croissant schema with `mlcroissant`.
- Examined available record sets and their fields by `@id`, then loaded the primary dataset into a pandas DataFrame.
- Performed basic filtering, normalization, and visualization of numeric data (e.g., latitude, longitude).
- Explored records grouped by key attributes (e.g., country or genus) where available.
- This analysis can be further extended to advanced modeling, clustering, and geospatial visualizations for biodiversity studies.
